<a href="https://colab.research.google.com/github/danielaschuck/data-engineering-studies/blob/main/ISOLATION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Niveis de Isolamento



Em bancos de dados, várias pessoas ou sistemas podem acessar e alterar informações ao mesmo tempo. Quando isso acontece, podem surgir problemas de concorrência, causando leituras incorretas ou inconsistentes dos dados. Para evitar esses problemas, os bancos de dados utilizam  níveis de isolamento, que controlam como uma transação pode visualizar as alterações feitas por outras transações. Neste material serão apresentados os principais problemas de concorrência: Dirty Read, Non-Repeatable Read e Phantom Read.



**Dirty Read**

O Dirty Read acontece quando uma transação lê um dado que ainda não foi confirmado por outra transação.
Ou seja, ela lê uma informação “suja”, que ainda pode ser desfeita.

No exemplo, a Transação 1 altera o saldo de uma conta de 1000 para 500, mas ainda não executa o `COMMIT`.

```python id="1hrltm"
cursor1.execute("BEGIN TRANSACTION")

cursor1.execute("""
UPDATE contas
SET saldo = 500
WHERE id = 1
""")
```

Nesse momento, a alteração ainda não foi salva definitivamente no banco.

Enquanto isso, a Transação 2 faz uma consulta e lê esse valor:

```python id="1f6sqn"
cursor2.execute("SELECT saldo FROM contas WHERE id = 1")
```

A Transação 2 encontra o saldo como 500, mesmo sem a confirmação da Transação 1.

Depois disso, a Transação 1 executa um `ROLLBACK`:

```python id="4r17nm"
conn1.rollback()
```

Isso significa que a alteração foi cancelada e o saldo volta para 1000.

O problema é que a Transação 2 leu um valor que nunca existiu oficialmente no banco, porque a alteração foi desfeita.
Por isso esse problema recebe o nome de Dirty Read, ou leitura suja.


In [1]:
import sqlite3

# conexão 1
conn1 = sqlite3.connect("teste.db")
cursor1 = conn1.cursor()

# conexão 2
conn2 = sqlite3.connect("teste.db")
cursor2 = conn2.cursor()

# criando tabela
cursor1.execute("""
CREATE TABLE IF NOT EXISTS contas (
    id INTEGER PRIMARY KEY,
    saldo INTEGER
)
""")

cursor1.execute("DELETE FROM contas")
cursor1.execute("INSERT INTO contas VALUES (1, 1000)")
conn1.commit()

# TRANSAÇÃO 1
cursor1.execute("BEGIN TRANSACTION")
cursor1.execute("""
UPDATE contas
SET saldo = 500
WHERE id = 1
""")

# TRANSAÇÃO 2 tenta ler
cursor2.execute("SELECT saldo FROM contas WHERE id = 1")

print("Saldo lido pela transação 2:", cursor2.fetchone()[0])

# desfaz alteração
conn1.rollback()

Saldo lido pela transação 2: 1000


Como evitar Dirty Read?

O Dirty Read pode ser evitado utilizando um nível de isolamento maior, como READ COMMITTED.

Nesse nível, uma transação só consegue ler dados que já foram confirmados com COMMIT.

Assim, alterações temporárias feitas por outras transações não ficam visíveis.

In [ ]:
SET TRANSACTION ISOLATION LEVEL READ COMMITTED;

BEGIN TRANSACTION;

SELECT *
FROM contas;

COMMIT;

O nível de isolamento é configurado antes da transação porque ele define as regras de leitura e concorrência que serão usadas durante toda a execução da transação.

Em muitos bancos, como PostgreSQL e MySQL, também existe um nível padrão já configurado.

Por exemplo:

PostgreSQL → READ COMMITTED já é o padrão.
MySQL/InnoDB → normalmente REPEATABLE READ.

**Non-Repeatable Read**


O ponto principal do Non-Repeatable Read é:

A mesma consulta retorna valores diferentes porque outra transação modificou os dados entre as leituras.



Uma transação deveria enxergar os dados de forma consistente do início ao fim.

O Non-Repeatable Read quebra essa consistência porque os dados mudam durante a execução.

In [ ]:
import sqlite3

conn1 = sqlite3.connect("teste.db")
conn2 = sqlite3.connect("teste.db")

c1 = conn1.cursor()
c2 = conn2.cursor()

# primeira leitura
c1.execute("SELECT saldo FROM contas WHERE id = 1")
print("Primeira leitura:", c1.fetchone()[0])

# outra transação altera
c2.execute("""
UPDATE contas
SET saldo = 1500
WHERE id = 1
""")
conn2.commit()

# segunda leitura
c1.execute("SELECT saldo FROM contas WHERE id = 1")
print("Segunda leitura:", c1.fetchone()[0])

**COMO EVITAR O NON REPEATABLE READ?**

ESTRUTURANDO COM **REPEATABLE READ**

Esse nível “congela” os registros lidos pela transação.

Assim, mesmo que outra transação altere os dados, a primeira continuará enxergando os mesmos valores até terminar.



In [ ]:
SET TRANSACTION ISOLATION LEVEL REPEATABLE READ;

BEGIN TRANSACTION;

SELECT saldo
FROM contas
WHERE id = 1;

-- outras operações

COMMIT;


**Phantom Read**


 “novas linhas aparecem” durante a transação.

**Por que isso é um problema?**

Porque a transação perde consistência.

Ela esperava trabalhar com o mesmo conjunto de registros, mas durante a execução surgiram novos dados.

Isso pode gerar:

relatórios inconsistentes;

contagens erradas;

falhas em cálculos;

problemas em sistemas financeiros.



In [ ]:
import sqlite3

conn1 = sqlite3.connect("teste.db")
conn2 = sqlite3.connect("teste.db")

c1 = conn1.cursor()
c2 = conn2.cursor()

# criando tabela
c1.execute("""
CREATE TABLE IF NOT EXISTS pedidos (
    id INTEGER,
    valor INTEGER
)
""")

c1.execute("DELETE FROM pedidos")

c1.execute("INSERT INTO pedidos VALUES (1, 120)")
c1.execute("INSERT INTO pedidos VALUES (2, 150)")
conn1.commit()

# primeira leitura
c1.execute("SELECT * FROM pedidos WHERE valor > 100")
print("Primeira leitura:", c1.fetchall())

# outra transação adiciona novo registro
c2.execute("INSERT INTO pedidos VALUES (3, 200)")
conn2.commit()

# segunda leitura
c1.execute("SELECT * FROM pedidos WHERE valor > 100")
print("Segunda leitura:", c1.fetchall())

COMO EVITAR O PHANTOM READ?

Usando o nivel de isolamento mais alto **SERIALIZABLE**

Ele impede que outras transações adicionem registros que afetem a consulta atual até a transação terminar.

In [ ]:
SET TRANSACTION ISOLATION LEVEL SERIALIZABLE;

BEGIN TRANSACTION;

SELECT *
FROM pedidos
WHERE valor > 100;

-- operações da transação

COMMIT;